In [ ]:
import os
import time
import pandas as pd
import numpy as np
import sys
import pathlib

cwd = pathlib.Path.cwd().resolve()
ansatz_pruning_dir = None
for candidate in [cwd, cwd / "AnsatzPruning", cwd.parent, cwd.parent / "AnsatzPruning"]:
    if (candidate / "AdaptVQE.py").exists():
        ansatz_pruning_dir = candidate
        break
if ansatz_pruning_dir is None:
    raise RuntimeError("Could not locate AnsatzPruning directory containing AdaptVQE.py.")
repo_root = ansatz_pruning_dir.parent

os.environ.setdefault("MPLCONFIGDIR", str(ansatz_pruning_dir / ".mplconfig"))
os.environ.setdefault("XDG_CACHE_HOME", str(ansatz_pruning_dir / ".cache"))
pathlib.Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
pathlib.Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import seaborn as sns

from qiskit.circuit import Parameter, QuantumCircuit
from qiskit.primitives import StatevectorEstimator

for p in [repo_root, ansatz_pruning_dir]:
    p_str = str(p)
    if p_str not in sys.path:
        sys.path.insert(0, p_str)

from MomentumMonteCarlo import momentum_sa_merged, momentum_sa_phased
from MomentumBuilder import MomentumBuilder as mb
from AnsatzBenchmarking.Problems.tsp.TSPProblems import TSPProblemSet
from AdaptVQE import (
    build_reference_state,
    build_operator_pool,
    run_adapt_vqe,
    extract_result_metrics,
)

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)


In [ ]:
TRIALS = 5
MB_ITERS = 3
SA_RUNS = 600
ADAPT_OPTIMIZER_MAXITER = 200
METHOD_ORDER = [
    "MomentumBuilder",
    "MomentumSA_Phased",
    "MomentumSA_Merged",
    "ADAPT-VQE",
]
METHOD_COLORS = {
    "MomentumBuilder": "#e74c3c",
    "MomentumSA_Phased": "#2ecc71",
    "MomentumSA_Merged": "#3498db",
    "ADAPT-VQE": "#BA8E23",
}
BUILDER_LABELS = {
    "MomentumBuilder": "MomentumBuilder",
    "momentum_sa_phased": "MomentumSA_Phased",
    "momentum_sa_merged": "MomentumSA_Merged",
}


def scalarize(value):
    if isinstance(value, np.ndarray):
        if value.ndim == 0:
            return float(np.real(value.item()))
        return float(np.real(value[-1]))
    if isinstance(value, list):
        return scalarize(np.array(value))
    return float(np.real(value))



def create_initial_components(num_qubits):
    circuit = QuantumCircuit(num_qubits)
    ansatz = QuantumCircuit(num_qubits)
    for qubit in range(num_qubits):
        ansatz.rx(Parameter(f"angle{qubit}"), qubit)
    return circuit, ansatz, [1.0] * num_qubits, list(range(num_qubits))



def evaluate_builder(builder, problems, trials=TRIALS):
    results = []
    estimator = StatevectorEstimator()
    problem_set = problems.getProblemSet()
    label = BUILDER_LABELS.get(builder.__name__, builder.__name__)

    for problem_idx, (h, exact) in enumerate(problem_set):
        print(f"\nProblem {problem_idx + 1}")

        for trial in range(trials):
            H = -h
            n_qubits = H.num_qubits
            circuit, ansatz, params_list, params_index = create_initial_components(n_qubits)

            start = time.time()
            if builder.__name__ == "MomentumBuilder":
                circuit, opt_params = builder(
                    params_list,
                    params_index,
                    ansatz,
                    circuit,
                    [*H.paulis, H],
                    estimator,
                    0.9,
                    0.99,
                    iters=MB_ITERS,
                    return_ansatz_and_params=True,
                )
                objective_energy = scalarize(
                    estimator.run([(circuit, H, opt_params)]).result()[0].data.evs
                )
            else:
                circuit, opt_params, objective_energy = builder(
                    params_list,
                    params_index,
                    ansatz,
                    circuit,
                    H,
                    estimator,
                    0.9,
                    0.99,
                    iters=MB_ITERS,
                    optimization_runs=SA_RUNS,
                )
                objective_energy = scalarize(objective_energy)
            build_time = time.time() - start

            displayed_energy = -objective_energy
            error = abs(displayed_energy - exact)

            print(
                f"Trial {trial + 1}/{trials}, Energy: {displayed_energy:.4f} "
                f"(Exact: {exact}), Time: {build_time:.2f}s"
            )

            results.append({
                "ansatz_type": label,
                "problem_index": problem_idx + 1,
                "trial": trial + 1,
                "time": build_time,
                "energy": objective_energy,
                "exact_energy": -exact,
                "error": error,
                "params": len(opt_params),
                "adapt_iterations": np.nan,
                "termination_criterion": None,
            })

    return results



def evaluate_adapt(problem_set, trials=TRIALS):
    results = []
    problems = problem_set.getProblemSet()

    for problem_idx, (h, exact) in enumerate(problems):
        print(f"\nProblem {problem_idx + 1}")

        for trial in range(trials):
            H = -h
            start = time.time()
            result = run_adapt_vqe(
                hamiltonian=H,
                initial_state=build_reference_state(H.num_qubits, angle=1.0),
                operator_pool=build_operator_pool(H.num_qubits),
                max_iterations=max(12, H.num_qubits),
                gradient_threshold=1e-6,
                eigenvalue_threshold=1e-8,
                optimizer_maxiter=ADAPT_OPTIMIZER_MAXITER,
            )
            elapsed = time.time() - start

            metrics = extract_result_metrics(result)
            objective_energy = scalarize(metrics["energy"])
            displayed_energy = -objective_energy
            error = abs(displayed_energy - exact)
            adapt_iterations = metrics.get("num_iterations")
            num_params = metrics.get("num_parameters")
            num_params = int(num_params) if num_params is not None else 0

            print(
                f"Trial {trial + 1}/{trials}, Energy: {displayed_energy:.4f} "
                f"(Exact: {exact}), Time: {elapsed:.2f}s, "
                f"Iters: {adapt_iterations}"
            )

            results.append({
                "ansatz_type": "ADAPT-VQE",
                "problem_index": problem_idx + 1,
                "trial": trial + 1,
                "time": elapsed,
                "energy": objective_energy,
                "exact_energy": -exact,
                "error": error,
                "params": num_params,
                "adapt_iterations": adapt_iterations,
                "termination_criterion": metrics.get("termination_criterion"),
            })

    return results


In [ ]:
# Load problem set and run benchmarks
print("Loading TSP problem set...")
problem_set = TSPProblemSet()
problems = problem_set.getProblemSet()
print(f"Loaded {len(problems)} TSP problems\n")

print("Running benchmarks...")
print("=" * 60)


In [ ]:
# Evaluate MomentumBuilder
print("\n" + "=" * 60)
print("MomentumBuilder:")
print("=" * 60)
momentum_results = evaluate_builder(mb, problem_set, trials=TRIALS)


In [ ]:
# Evaluate ADAPT-VQE
print("\n" + "=" * 60)
print("ADAPT-VQE:")
print("=" * 60)
adapt_results = evaluate_adapt(problem_set, trials=TRIALS)


In [ ]:
# Evaluate phasedMomentumBuilder
print("\n" + "=" * 60)
print("Phased MomentumBuilder:")
print("=" * 60)
phased_momentum_results = evaluate_builder(momentum_sa_phased, problem_set, trials=TRIALS)


In [ ]:
# Evaluate mergedMomentumBuilder
print("\n" + "=" * 60)
print("Merged MomentumBuilder:")
print("=" * 60)
merged_momentum_results = evaluate_builder(momentum_sa_merged, problem_set, trials=TRIALS)


In [ ]:
# Combine results
all_results = momentum_results + phased_momentum_results + merged_momentum_results + adapt_results
df = pd.DataFrame(all_results)
if len(df) > 0:
    df["ansatz_type"] = pd.Categorical(df["ansatz_type"], categories=METHOD_ORDER, ordered=True)
    df = df.sort_values(["ansatz_type", "problem_index", "trial"]).reset_index(drop=True)
print(f"\n{'=' * 60}")
print(f"Completed: {len(df)} successful trials")


In [ ]:
# Display summary statistics
if len(df) > 0:
    print("\n" + "=" * 60)
    print("SUMMARY STATISTICS")
    print("=" * 60)

    summary = df.groupby("ansatz_type", observed=False).agg({
        "energy": ["mean", "std", "min", "max"],
        "time": ["mean", "std", "min", "max"],
        "error": ["mean", "std", "min", "max"],
        "params": ["mean", "std"],
    }).reindex(METHOD_ORDER).dropna(how="all")

    print("Energy")
    print(summary["energy"].to_string())
    print("\nTime")
    print(summary["time"].to_string())
    print("\nError")
    print(summary["error"].to_string())
    print("\nParameters")
    print(summary["params"].to_string())

    adapt_only = df[df["ansatz_type"] == "ADAPT-VQE"]
    if len(adapt_only) > 0:
        print("\nADAPT diagnostics")
        print(
            adapt_only[["adapt_iterations"]]
            .agg(["mean", "std", "min", "max"])
            .to_string()
        )
        print(
            "Termination criteria:",
            sorted(adapt_only["termination_criterion"].dropna().unique()),
        )
else:
    print("No successful trials to display.")


In [ ]:
# Visualization: comparison plots
if len(df) > 0 and "ansatz_type" in df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    available = [label for label in METHOD_ORDER if label in set(df["ansatz_type"].astype(str))]

    ax1 = axes[0, 0]
    energy_data = [df[df["ansatz_type"] == label]["energy"].dropna().values for label in available]
    bplot1 = ax1.boxplot(energy_data, tick_labels=available, patch_artist=True)
    for patch, label in zip(bplot1["boxes"], available):
        patch.set_facecolor(METHOD_COLORS[label])
        patch.set_alpha(0.5)
    ax1.set_title("Final Energy Comparison", fontsize=14, fontweight="bold")
    ax1.set_xlabel("Ansatz Type")
    ax1.set_ylabel("Energy")
    ax1.grid(True, alpha=0.3)

    ax2 = axes[0, 1]
    time_data = [df[df["ansatz_type"] == label]["time"].dropna().values for label in available]
    bplot2 = ax2.boxplot(time_data, tick_labels=available, patch_artist=True)
    for patch, label in zip(bplot2["boxes"], available):
        patch.set_facecolor(METHOD_COLORS[label])
        patch.set_alpha(0.5)
    ax2.set_title("Optimization Time Comparison", fontsize=14, fontweight="bold")
    ax2.set_xlabel("Ansatz Type")
    ax2.set_ylabel("Time (seconds)")
    ax2.grid(True, alpha=0.3)

    ax3 = axes[1, 0]
    param_means = df.groupby("ansatz_type", observed=False)["params"].mean().reindex(available).dropna()
    param_stds = df.groupby("ansatz_type", observed=False)["params"].std().reindex(param_means.index).fillna(0.0)
    x_pos = np.arange(len(param_means))
    ax3.bar(
        x_pos,
        param_means.values,
        yerr=param_stds.values,
        color=[METHOD_COLORS[label] for label in param_means.index],
        alpha=0.7,
        capsize=5,
    )
    ax3.set_title("Number of Parameters", fontsize=14, fontweight="bold")
    ax3.set_xlabel("Ansatz Type")
    ax3.set_ylabel("Parameters")
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels(param_means.index, rotation=15, ha="right")
    ax3.grid(True, alpha=0.3, axis="y")

    ax4 = axes[1, 1]
    for label in available:
        subset = df[df["ansatz_type"] == label]
        ax4.scatter(
            subset["time"],
            subset["energy"],
            label=label,
            color=METHOD_COLORS[label],
            alpha=0.7,
            s=90,
        )
    ax4.set_title("Energy vs Time Trade-off", fontsize=14, fontweight="bold")
    ax4.set_xlabel("Time (seconds)")
    ax4.set_ylabel("Energy")
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("ansatz_comparison.png", dpi=300, bbox_inches="tight")
    plt.show()
